# SDC-2022 Multipath — Data Parser

Parses the Google Smartphone Decimeter Challenge **2022** training logs and uses each device's own `MultipathIndicator` field (0 = clean / not-flagged, 1 = multipath detected) as the ground-truth label — exactly the same label source used in the Mi8 pipeline. Because the label is reported per raw measurement by the receiver, **no SPAN time-sync is required**.

Only some device models in this dataset actually populate `MultipathIndicator = 1` (in the 2022 sessions: **mi8, pixel6pro, pixel7, pixel7pro, sm-g988b, samsungs21ultra**); every other device reports a constant 0, which means *not-populated* rather than *confirmed clean*. Including those would poison the clean class with unknown-status measurements, so this notebook **keeps only the device files that report at least one multipath flag**.

**Input:** `data/01_raw/sdc2023/train/2022-*/<device>/device_gnss.csv`

**Output:** `data/02_interim/sdc2022_epochs.csv`

## 1. Setup & Configuration

In [1]:
import os
import glob
import pandas as pd
import numpy as np

BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '../..'))
RAW_DIR    = os.path.join(BASE_DIR, 'data/01_raw/sdc2023/train')
OUTPUT_DIR = os.path.join(BASE_DIR, 'data/02_interim')
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'sdc2022_epochs.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Every device_gnss.csv under a 2022 session
GNSS_FILES = sorted(glob.glob(os.path.join(RAW_DIR, '2022-*', '*', 'device_gnss.csv')))
print(f'Found {len(GNSS_FILES)} device_gnss.csv files under 2022 sessions')

Found 26 device_gnss.csv files under 2022 sessions


## 2. Select the Multipath-Reporting Device Files

Scan every 2022 device file and keep only those whose `MultipathIndicator` column contains at least one `1`. These are the only files where the label is a genuine hardware determination rather than a constant placeholder.

In [2]:
def session_device(path):
    parts = path.replace('\\', '/').split('/')
    return parts[-3], parts[-2]   # session folder, device folder

selected = []
print(f'{"session":38} {"device":16} {"rows":>8} {"MP=1":>7}')
for f in GNSS_FILES:
    sess, dev = session_device(f)
    mp = pd.read_csv(f, usecols=['MultipathIndicator'])['MultipathIndicator']
    n1 = int((mp == 1).sum())
    if n1 > 0:
        selected.append((f, sess, dev))
        print(f'{sess:38} {dev:16} {len(mp):>8} {n1:>7}')

print(f'\nSelected {len(selected)} device files '
      f'across {len({s for _, s, _ in selected})} unique sessions.')

session                                device               rows    MP=1
2022-01-11-18-48-us-ca-mtv-n           mi8                 42396    4979


2022-01-11-18-48-us-ca-mtv-n           pixel6pro           28737    2009
2022-01-26-20-02-us-ca-mtv-pe1         mi8                 75495    3925


2022-01-26-20-02-us-ca-mtv-pe1         sm-g988b            68939    7328


2022-02-24-18-29-us-ca-lax-o           mi8                114467   11460


2022-02-24-18-29-us-ca-lax-o           pixel6pro           72178    6318
2022-04-01-18-22-us-ca-lax-t           mi8                 64043    4223


2022-04-01-18-22-us-ca-lax-t           pixel6pro           52114    7287
2022-05-13-20-57-us-ca-mtv-pe1         pixel6pro           63333    5713


2022-05-13-20-57-us-ca-mtv-pe1         samsungs21ultra     96853   12158


2022-05-13-20-57-us-ca-mtv-pe1         sm-g988b            86933   10711


2022-08-04-20-07-us-ca-sjc-q           mi8                 63852    3840


2022-11-15-00-53-us-ca-mtv-a           pixel7              35670    1032
2022-11-15-00-53-us-ca-mtv-a           pixel7pro           35747    1444

Selected 14 device files across 7 unique sessions.


## 3. Parse Raw GNSS Measurements

The 2022 `device_gnss.csv` is already a clean CSV (unlike the raw Mi8 `GnssLog.txt`), so parsing is a direct read. Alongside the signal-quality features shared with the Mi8 pipeline, the 2022 format also provides **satellite geometry** (elevation / azimuth), the **raw pseudorange**, and the columns needed to compute a **pseudorange residual** — all strong physical multipath indicators.

In [3]:
# Signal-quality + geometry features we keep as-is
FEATURE_COLS = [
    'utcTimeMillis', 'TimeNanos', 'FullBiasNanos', 'State', 'Svid',
    'ReceivedSvTimeUncertaintyNanos',
    'Cn0DbHz', 'BasebandCn0DbHz', 'SnrInDb', 'AgcDb',
    'PseudorangeRateMetersPerSecond', 'PseudorangeRateUncertaintyMetersPerSecond',
    'AccumulatedDeltaRangeState', 'AccumulatedDeltaRangeMeters',
    'AccumulatedDeltaRangeUncertaintyMeters',
    'CarrierFrequencyHz', 'ConstellationType', 'MultipathIndicator',
    'SvElevationDegrees', 'SvAzimuthDegrees',
    'RawPseudorangeMeters', 'RawPseudorangeUncertaintyMeters',
]
# Extra columns used ONLY to derive the pseudorange residual, then dropped
RESIDUAL_INPUTS = [
    'SvPositionXEcefMeters', 'SvPositionYEcefMeters', 'SvPositionZEcefMeters',
    'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters',
    'SvClockBiasMeters', 'IsrbMeters',
    'IonosphericDelayMeters', 'TroposphericDelayMeters',
]

def parse_gnss(path, sess, dev):
    df = pd.read_csv(path, low_memory=False)
    keep = [c for c in FEATURE_COLS + RESIDUAL_INPUTS if c in df.columns]
    df = df[keep].copy()
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['session'] = sess
    df['device']  = dev
    return df

frames = []
for f, sess, dev in selected:
    df = parse_gnss(f, sess, dev)
    frames.append(df)
    print(f'  {sess} / {dev}: {len(df):,} rows')

raw_df = pd.concat(frames, ignore_index=True)
print(f'\nTotal raw measurements: {len(raw_df):,}')
raw_df.head()

  2022-01-11-18-48-us-ca-mtv-n / mi8: 42,396 rows


  2022-01-11-18-48-us-ca-mtv-n / pixel6pro: 28,737 rows


  2022-01-26-20-02-us-ca-mtv-pe1 / mi8: 75,495 rows


  2022-01-26-20-02-us-ca-mtv-pe1 / sm-g988b: 68,939 rows


  2022-02-24-18-29-us-ca-lax-o / mi8: 114,467 rows


...

  2022-05-13-20-57-us-ca-mtv-pe1 / sm-g988b: 86,933 rows


  2022-08-04-20-07-us-ca-sjc-q / mi8: 63,852 rows


  2022-11-15-00-53-us-ca-mtv-a / pixel7: 35,670 rows


  2022-11-15-00-53-us-ca-mtv-a / pixel7pro: 35,747 rows

Total raw measurements: 900,757


,utcTimeMillis,TimeNanos,FullBiasNanos,State,Svid,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,BasebandCn0DbHz,SnrInDb,AgcDb,...,SvPositionZEcefMeters,WlsPositionXEcefMeters,WlsPositionYEcefMeters,WlsPositionZEcefMeters,SvClockBiasMeters,IsrbMeters,IonosphericDelayMeters,TroposphericDelayMeters,session,device
0,1641926922000,173758000000,-1325961966242366384,16392,2,1000000000,7.000000,NaN,NaN,NaN,...,NaN,-2.682707e+06,-4.311631e+06,3.846284e+06,NaN,NaN,NaN,NaN,2022-01-11-18-48-us-ca-mtv-n,mi8
1,1641926922000,173758000000,-1325961966242366384,47,3,17,35.867416,NaN,NaN,NaN,...,-1.461030e+05,-2.682707e+06,-4.311631e+06,3.846284e+06,-22321.104528,0.0,4.038903,3.725371,2022-01-11-18-48-us-ca-mtv-n,mi8
2,1641926922000,173758000000,-1325961966242366384,47,4,12,42.319908,NaN,NaN,NaN,...,1.934849e+07,-2.682707e+06,-4.311631e+06,3.846284e+06,-58576.459585,0.0,2.890557,2.662006,2022-01-11-18-48-us-ca-mtv-n,mi8
3,1641926922000,173758000000,-1325961966242366384,47,6,21,33.128010,NaN,NaN,NaN,...,9.540529e+06,-2.682707e+06,-4.311631e+06,3.846284e+06,50384.884950,0.0,5.825073,10.834992,2022-01-11-18-48-us-ca-mtv-n,mi8
4,1641926922000,173758000000,-1325961966242366384,47,7,15,37.043587,NaN,NaN,NaN,...,8.309226e+06,-2.682707e+06,-4.311631e+06,3.846284e+06,90165.671589,0.0,4.108635,4.217873,2022-01-11-18-48-us-ca-mtv-n,mi8


## 4. Outlier Rejection

Applies Google's official quality filters (identical to the Mi8 pipeline): valid non-zero `FullBiasNanos`, positive `TimeNanos`, a decoded/known time-of-week `State` bit, and code-lock timing uncertainty ≤ 500 ns.

In [4]:
def apply_outlier_rejection(df):
    n0 = len(df)
    df = df.dropna(subset=['FullBiasNanos', 'TimeNanos'])
    df = df[(df['FullBiasNanos'] != 0) & (df['TimeNanos'] > 0)]
    state = df['State'].fillna(0).astype('int64')
    state_ok = ((state & (1 << 3)) != 0) | ((state & (1 << 14)) != 0)
    df = df[state_ok]
    df = df[df['ReceivedSvTimeUncertaintyNanos'] <= 500]
    print(f'Rejected {n0 - len(df):,} rows  |  Kept {len(df):,} rows')
    return df

clean_df = apply_outlier_rejection(raw_df.copy())
clean_df.head()

Rejected 280,121 rows  |  Kept 620,636 rows


,utcTimeMillis,TimeNanos,FullBiasNanos,State,Svid,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,BasebandCn0DbHz,SnrInDb,AgcDb,...,SvPositionZEcefMeters,WlsPositionXEcefMeters,WlsPositionYEcefMeters,WlsPositionZEcefMeters,SvClockBiasMeters,IsrbMeters,IonosphericDelayMeters,TroposphericDelayMeters,session,device
1,1641926922000,173758000000,-1325961966242366384,47,3,17,35.867416,NaN,NaN,NaN,...,-1.461030e+05,-2.682707e+06,-4.311631e+06,3.846284e+06,-22321.104528,0.0,4.038903,3.725371,2022-01-11-18-48-us-ca-mtv-n,mi8
2,1641926922000,173758000000,-1325961966242366384,47,4,12,42.319908,NaN,NaN,NaN,...,1.934849e+07,-2.682707e+06,-4.311631e+06,3.846284e+06,-58576.459585,0.0,2.890557,2.662006,2022-01-11-18-48-us-ca-mtv-n,mi8
3,1641926922000,173758000000,-1325961966242366384,47,6,21,33.128010,NaN,NaN,NaN,...,9.540529e+06,-2.682707e+06,-4.311631e+06,3.846284e+06,50384.884950,0.0,5.825073,10.834992,2022-01-11-18-48-us-ca-mtv-n,mi8
4,1641926922000,173758000000,-1325961966242366384,47,7,15,37.043587,NaN,NaN,NaN,...,8.309226e+06,-2.682707e+06,-4.311631e+06,3.846284e+06,90165.671589,0.0,4.108635,4.217873,2022-01-11-18-48-us-ca-mtv-n,mi8
6,1641926922000,173758000000,-1325961966242366384,47,9,14,37.668007,NaN,NaN,NaN,...,2.160265e+07,-2.682707e+06,-4.311631e+06,3.846284e+06,-107389.639498,0.0,3.168356,3.083715,2022-01-11-18-48-us-ca-mtv-n,mi8


## 5. Compute GPS Time of Week

Derives `GpsTimeNanos` (nanoseconds since the start of the current GPS week) from `TimeNanos` and `FullBiasNanos`, giving an absolute time reference for ordering and per-epoch grouping.

In [5]:
NANOS_PER_SECOND = 1e9
SECONDS_PER_WEEK = 604800

raw_gps_ns = clean_df['TimeNanos'].astype('int64') - clean_df['FullBiasNanos'].astype('int64')
clean_df['GpsTimeNanos'] = raw_gps_ns % int(SECONDS_PER_WEEK * NANOS_PER_SECOND)
print('GpsTimeNanos range:'
      f"  min={clean_df['GpsTimeNanos'].min():,}  max={clean_df['GpsTimeNanos'].max():,}")
clean_df[['session', 'device', 'GpsTimeNanos', 'Cn0DbHz', 'SvElevationDegrees', 'MultipathIndicator']].head()

GpsTimeNanos range:  min=176,047,999,523,889  max=509,621,999,899,575


,session,device,GpsTimeNanos,Cn0DbHz,SvElevationDegrees,MultipathIndicator
1,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,35.867416,42.065444,0
2,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,42.319908,69.824656,0
3,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,33.128010,13.110256,0
4,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,37.043587,36.256790,0
6,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,37.668007,54.089851,0


## 6. Compute Pseudorange Residual

The single most physically-direct multipath signature is the **pseudorange residual** — the difference between the measured raw pseudorange and the geometric range implied by the satellite and receiver positions.

For each measurement:

```
geometric_range = || SvPosition_ECEF  -  WlsPosition_ECEF ||
modeled_range   = geometric_range  - SvClockBias  + Iono + Tropo + Isrb
residual_raw    = RawPseudorange    - modeled_range
```

`residual_raw` still contains the receiver clock bias, which is **common to every satellite in the same epoch**. Subtracting the per-epoch median removes it, leaving the per-satellite error — dominated by multipath and thermal noise. A large `|prResidual|` is a classic multipath tell.

In [6]:
geo_cols_present = all(c in clean_df.columns for c in [
    'SvPositionXEcefMeters', 'WlsPositionXEcefMeters', 'RawPseudorangeMeters'])

if geo_cols_present:
    sv = clean_df[['SvPositionXEcefMeters', 'SvPositionYEcefMeters', 'SvPositionZEcefMeters']].to_numpy()
    rx = clean_df[['WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']].to_numpy()
    geo = np.sqrt(((sv - rx) ** 2).sum(axis=1))
    modeled = (geo
               - clean_df['SvClockBiasMeters'].fillna(0)
               + clean_df['IonosphericDelayMeters'].fillna(0)
               + clean_df['TroposphericDelayMeters'].fillna(0)
               + clean_df['IsrbMeters'].fillna(0))
    res_raw = clean_df['RawPseudorangeMeters'] - modeled
    clean_df['_res_raw'] = res_raw
    # de-mean the receiver clock bias per (session, device, epoch)
    epoch_median = clean_df.groupby(['session', 'device', 'utcTimeMillis'])['_res_raw'].transform('median')
    clean_df['prResidual'] = clean_df['_res_raw'] - epoch_median
    clean_df.drop(columns=['_res_raw'], inplace=True)
    print('prResidual computed.')
    print(clean_df['prResidual'].describe())
else:
    print('Geometry columns missing — skipping prResidual.')

# Drop the raw geometry/model inputs now that the residual is derived
clean_df.drop(columns=[c for c in RESIDUAL_INPUTS if c in clean_df.columns], inplace=True)

prResidual computed.
count    5.920040e+05
mean    -7.084901e+00
std      6.388326e+03
min     -4.071184e+06
25%     -1.499342e+01
50%      0.000000e+00
75%      1.415358e+01
max      1.329911e+06
Name: prResidual, dtype: float64


## 7. Label Distribution

How much multipath was flagged overall and per session/device.

In [7]:
total = len(clean_df)
dist  = clean_df['MultipathIndicator'].value_counts().sort_index()
print('=== Overall MultipathIndicator distribution ===')
for val, cnt in dist.items():
    print(f'  {int(val)} : {cnt:>9,}  ({100*cnt/total:.1f}%)')

print('\n=== Per session / device ===')
grp = (clean_df.groupby(['session', 'device', 'MultipathIndicator']).size()
       .unstack(fill_value=0).rename(columns={0: 'Clean', 1: 'Multipath'}))
if 'Multipath' in grp.columns:
    grp['Multipath_%'] = (100 * grp['Multipath'] / grp.sum(axis=1)).round(1)
print(grp)

=== Overall MultipathIndicator distribution ===
  0 :   580,812  (93.6%)
  1 :    39,824  (6.4%)

=== Per session / device ===


MultipathIndicator                              Clean  Multipath  Multipath_%
session                        device                                        
2022-01-11-18-48-us-ca-mtv-n   mi8              25435       1558          5.8
                               pixel6pro        20811       1032          4.7
2022-01-26-20-02-us-ca-mtv-pe1 mi8              51467       1618          3.0
                               sm-g988b         48911       3855          7.3
2022-02-24-18-29-us-ca-lax-o   mi8              61035       4802          7.3
                               pixel6pro        44545       3148          6.6
2022-04-01-18-22-us-ca-lax-t   mi8              40483       2310          5.4
                               pixel6pro        31908       3517          9.9
2022-05-13-20-57-us-ca-mtv-pe1 pixel6pro        41916       4390          9.5
                               samsungs21ultra  59128       5360          8.3
                               sm-g988b         62148       4846

## 8. Export

Drops the raw time columns that are not predictive features and writes the interim epoch table.

In [8]:
final_df = clean_df.drop(columns=['TimeNanos', 'FullBiasNanos', 'utcTimeMillis'], errors='ignore').copy()
final_df.dropna(subset=['MultipathIndicator'], inplace=True)
final_df['MultipathIndicator'] = final_df['MultipathIndicator'].astype(int)

final_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(final_df):,} rows  ->  {OUTPUT_CSV}')
print(f'Columns: {list(final_df.columns)}')
final_df.head()

Saved 620,636 rows  ->  C:\Users\Dell\Documents\Warwick\Diss\Code\Dissertation_Trial\GNSS_Multipath_Project\data/02_interim\sdc2022_epochs.csv
Columns: ['State', 'Svid', 'ReceivedSvTimeUncertaintyNanos', 'Cn0DbHz', 'BasebandCn0DbHz', 'SnrInDb', 'AgcDb', 'PseudorangeRateMetersPerSecond', 'PseudorangeRateUncertaintyMetersPerSecond', 'AccumulatedDeltaRangeState', 'AccumulatedDeltaRangeMeters', 'AccumulatedDeltaRangeUncertaintyMeters', 'CarrierFrequencyHz', 'ConstellationType', 'MultipathIndicator', 'SvElevationDegrees', 'SvAzimuthDegrees', 'RawPseudorangeMeters', 'RawPseudorangeUncertaintyMeters', 'session', 'device', 'GpsTimeNanos', 'prResidual']


,State,Svid,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,BasebandCn0DbHz,SnrInDb,AgcDb,PseudorangeRateMetersPerSecond,PseudorangeRateUncertaintyMetersPerSecond,AccumulatedDeltaRangeState,...,ConstellationType,MultipathIndicator,SvElevationDegrees,SvAzimuthDegrees,RawPseudorangeMeters,RawPseudorangeUncertaintyMeters,session,device,GpsTimeNanos,prResidual
1,47,3,17,35.867416,NaN,NaN,NaN,594.420187,0.002924,17,...,1,0,42.065444,175.396281,2.193652e+07,5.096472,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,-31.072611
2,47,4,12,42.319908,NaN,NaN,NaN,48.766446,0.001466,17,...,1,0,69.824656,46.728665,2.053317e+07,3.597509,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,-20.017892
3,47,6,21,33.128010,NaN,NaN,NaN,302.023892,0.004009,17,...,1,0,13.110256,277.077207,2.439315e+07,6.295642,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,161.380937
4,47,7,15,37.043587,NaN,NaN,NaN,-368.476984,0.002554,17,...,1,0,36.256790,255.396843,2.257167e+07,4.496887,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,2.367246
6,47,9,14,37.668007,NaN,NaN,NaN,-328.389457,0.002377,17,...,1,0,54.089851,318.827182,2.119126e+07,4.197094,2022-01-11-18-48-us-ca-mtv-n,mi8,240540000366384,-2.308638
